### Ingest Result.json file
1. Read the file using spark dataframe reader API
2. Define and enforce schema
3. Add metadata columns
   - Source File
    - ingestion timestamp
3. Write to bronze delta table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType

results_schema = StructType(fields=[
    StructField("date",DateType()),
    StructField("raceName",StringType()),
    StructField("season",IntegerType()),
    StructField("driverId",StringType()),
    StructField("round",IntegerType()),
    StructField("constructorId",StringType()),
    StructField("url",StringType()),
    StructField("grid",IntegerType()),
    StructField("laps",IntegerType()),
    StructField("number",IntegerType()),
    StructField("points",FloatType()),
    StructField("position",IntegerType()),
    StructField("positionText",StringType()),
    StructField("status",StringType()),
])


In [0]:
results_df = (
    spark.read.format('json')
    .schema(results_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
display(results_df)

In [0]:
results_final_df = add_ingestion_metadata(results_df)

In [0]:
%sql
select season,count(*) from formula1.bronze.results
group by season
order by season 

In [0]:
(
    results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)